# Assignment II: Rate Cuts versus Stimulus Payments

One section per part. Complete the cells where it says `write your code here`. All plotting and printing code is given: use the provided plotting and printing functions for all figures and tables.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from HANKModel import HANKModelClass

## Settings

In [ ]:
plt.rcParams.update({'axes.grid':True,'grid.alpha':0.3,'figure.figsize':(6,4)})
os.makedirs('figs',exist_ok=True)

## 1. The Intertemporal Keynesian Cross

Part 1 is pen and paper.

## 2. RANK

Part 2 is pen and paper.

## 3. HANK

The model code in `blocks.py` is given.

In [ ]:
model = HANKModelClass(name='wolf')
model.find_ss(do_print=True)

par = model.par
ss = model.ss

q = (1+ss.r)**(-np.arange(par.T))

### 3(a) The Jacobians

In [ ]:
model.compute_jacs()

# write your code here: M, M_chi and M_ra from model.jac_hh (see Section 3 of the assignment)
M = np.nan
M_chi = np.nan
M_ra = np.nan

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(15,4))
for s in [0,5,10,20]:
    axes[0].plot(M_ra[:30,s],label=f's = {s}')
    axes[1].plot(M[:30,s],label=f's = {s}')
    axes[2].plot(M_chi[:30,s],label=f's = {s}')
axes[0].set_title(r'columns of $M^{ra}$')
axes[1].set_title(r'columns of $M$')
axes[2].set_title(r'columns of $M^\chi$')
for ax in axes:
    ax.set_xlabel('years')
axes[0].legend()
fig.tight_layout()
fig.savefig('figs/jacobians.pdf')

### 3(b) The rate cut

In [ ]:
model.find_IRFs(shocks=['eps_r'])
IRF_r = deepcopy(model.IRF)
dra = IRF_r['ra']

dY_IKC = np.nan # write your code here

print(f'output on impact = {100*IRF_r["Y"][0]:.3f}%')
print(f'largest gap between the IKC and find_IRFs = {np.abs(dY_IKC-IRF_r["Y"]).max():.1e}')

### 3(c) The transfer shock

In [ ]:
print(f'M_chi[0,0] = {M_chi[0,0]:.3f}, 1-beta in RANK = {ss.r/(1+ss.r):.4f}')
print(f'condition number of M_chi = {np.linalg.cond(M_chi):.1f}')

dchi = np.nan # write your code here

model.find_IRFs(shocks={'dchi':dchi})
IRF_chi = deepcopy(model.IRF)

titles = {'r':'real rate (bp)','chi':'transfer (% of Y)','tau':'tax rate (pp)','B':'debt (% of Y)','Y':'output (%)','pi':'inflation (pp)'}
scales = {'r':1e4,'chi':100,'tau':100,'B':100,'Y':100,'pi':100}

fig,axes = plt.subplots(2,3,figsize=(12,6))
for ax,(var,title) in zip(axes.flatten(),titles.items()):
    ax.plot(scales[var]*IRF_r[var][:12],label='rate cut')
    ax.plot(scales[var]*IRF_chi[var][:12],ls='--',label='transfer shock')
    ax.set_title(title)
    ax.set_xlabel('years')
axes[0,0].legend()
fig.tight_layout()
fig.savefig('figs/shocks.pdf')

print(f'largest gap in output = {np.abs(IRF_r["Y"]-IRF_chi["Y"]).max():.1e}')
print(f'largest gap in inflation = {np.abs(IRF_r["pi"]-IRF_chi["pi"]).max():.1e}')

### 3(d) The size of the transfers

In [ ]:
print(f'peak output = {100*IRF_r["Y"].max():.3f}%')
print(f'peak transfer shock = {100*dchi.max():.3f}% of output')
print(f'first year with a negative transfer shock = {np.argmax(dchi<0)}')
print(f'peak debt = {100*IRF_chi["B"].max():.3f}% of output, in year {IRF_chi["B"].argmax()}')
print(f'present value of the transfer shock = {q@dchi:.1e}')

## 4. Welfare

In [ ]:
def quintile_distributions(model):
    """ initial distribution of each steady-state wealth quintile, each with mass one """

    par = model.par
    ss = model.ss

    # a. sort household states by wealth
    order = np.argsort(np.tile(par.a_grid,par.Nz),kind='stable')
    cumulative_mass = np.r_[0,np.cumsum(ss.Dbeg[0].ravel()[order])]

    # b. mass of each state in each quintile, splitting a state between two quintiles if needed
    Dbegs = []
    for i in range(5):
        mass = np.zeros(par.Nz*par.Na)
        mass[order] = np.diff(np.clip(cumulative_mass,i/5,(i+1)/5))
        Dbegs.append(5*mass.reshape(ss.Dbeg.shape))

    return Dbegs

def welfare(model,Dbeg,IRF=None):
    """ discounted sum of aggregate utility of the households who start in Dbeg, in steady state or after a shock """

    par = model.par
    ss = model.ss

    # a. path of the households, with all inputs at steady state or with the responses to a shock
    if IRF is None:
        path = model.decompose_hh_path(Dbeg=Dbeg)
    else:
        custom_paths = {'Z':ss.Z+IRF['Z'],'ra':ss.ra+IRF['ra'],'chi':ss.chi+IRF['chi']}
        path = np.nan # write your code here: use model.decompose_hh_path with Dbeg, use_inputs and custom_paths

    # b. discounted sum of aggregate utility
    W = np.nan # write your code here: use path.U_hh

    return W

def consumption_equivalent(W_shock,W_ss,model):
    """ permanent percent rise in consumption that gives the steady state the welfare of the shock """

    par = model.par

    return 100*((W_shock/W_ss)**(1/(1-par.sigma))-1)

In [ ]:
Dbegs = quintile_distributions(model)+[ss.Dbeg]

welfare_table = pd.DataFrame(index=['Q1','Q2','Q3','Q4','Q5','all'],columns=['rate cut','transfer shock'],dtype=float)
for name,Dbeg in zip(welfare_table.index,Dbegs):
    W_ss = welfare(model,Dbeg)
    welfare_table.loc[name,'rate cut'] = consumption_equivalent(welfare(model,Dbeg,IRF_r),W_ss,model)
    welfare_table.loc[name,'transfer shock'] = consumption_equivalent(welfare(model,Dbeg,IRF_chi),W_ss,model)

fig,ax = plt.subplots()
welfare_table.plot.bar(ax=ax,rot=0)
ax.set_xlabel('wealth quintile')
ax.set_title('welfare, consumption equivalent (%)')
fig.tight_layout()
fig.savefig('figs/welfare.pdf')

welfare_table.round(4)

## 5. Conclusion

Question 5 needs no code.

## 6. Capital gains

Question 6 needs no code.